#### Construct representative test set

- competition test set is very short (10 days) and affected by Storm Amy
- could use a more representative test set instead of length _num_ until day _cut_off_
- not used in thesis


In [1]:
import sys

sys.path.append("../src")

from datetime import datetime
import glob
from types import SimpleNamespace

from configs.load_config import load_config
from data_processing.load import (
    load_evaluation_data,
    load_evaluation_forecast_data,
    load_scaler_params,
)

In [2]:
def extract_test_dates(num=30, data_path="../data/evaluation/", cut_off="2025-09-22"):
    """
    Evaluation dates (#num) for representative test set, cut_off determines until which date the data can serve as test set
    """
    extract_date_weather = lambda f: datetime.strptime(
        f.split("/")[-1].split("_")[2][:8], "%Y%m%d"
    )
    extract_date_power = lambda f: datetime.strptime(
        f.split("/")[-1].split("_")[-1].replace(".parquet", ""), "%Y%m%d"
    )

    files_weather = glob.glob(data_path + "/met_forecast_daily/*.csv")
    files_power = glob.glob(data_path + "/wind_power/*.parquet")

    dates_weather = [extract_date_weather(f) for f in files_weather]
    dates_weather = sorted(dates_weather)
    dates_power = [extract_date_power(f) for f in files_power]
    dates_power = sorted(dates_power)

    date_strings_weather = [d.strftime("%Y-%m-%d") for d in dates_weather]
    date_strings_power = [d.strftime("%Y-%m-%d") for d in dates_power]

    dates = [date for date in date_strings_weather if date in date_strings_power]

    actual_test_set_idx = dates.index(cut_off)
    valid_dates = dates[:actual_test_set_idx]
    return valid_dates[-num:]

In [3]:
model_cfg = load_config("../configs/model/bnn_gen_model_nowcast_02.yaml")
file_name = "WindPowerQuantile_weighted_2026-03-09_16-35-16_nowcast_gen_ELSPOT_NO3"
data_cfg = load_config(model_cfg.data.path)
data_cfg.dataset = "forecast"
data_cfg.ensemble = SimpleNamespace(members=None, variance=False)

evaluation_dates = extract_test_dates(cut_off="2025-09-22")

In [4]:
bid_zone = model_cfg.data.bid_zone

# Scale nowcast and forecast with same scalers
scaler_data_nowcast = load_scaler_params(
    bid_zone=bid_zone, dataset="nowcast", dir_path="../checkpoints"
)

if data_cfg.dataset == "nowcast":

    eval_data = load_evaluation_data(
        dates=evaluation_dates,
        scaler_params=scaler_data_nowcast["scaler_params"],
        site_scaler_params=scaler_data_nowcast["site_scaler_params"],
        bid_zone=bid_zone,
    )

elif data_cfg.dataset == "forecast":

    eval_data = load_evaluation_forecast_data(
        dates=evaluation_dates,
        scaler_params=scaler_data_nowcast["scaler_params"],
        site_scaler_params=scaler_data_nowcast["site_scaler_params"],
        bid_zone=bid_zone,
    )

Loaded scaler parameters for ELSPOT NO2 and nowcast
  Saved at: 2026-04-07 15:24:08
  Split time: 2025-01-01 00:00:00+01:00
Loading evaluation data for 30 dates in zone ELSPOT NO2...
Loading evaluation data for 2025-08-21 in zone ELSPOT NO2
Dropping 15 gust columns not present in nowcast

Column mapping summary:
  Original columns: 94
  Renamed columns: 79
  Dropped columns: 15
  Final columns: 79

 RangeIndex(start=0, stop=4087, step=1)
Loading evaluation data for 2025-08-22 in zone ELSPOT NO2
Dropping 15 gust columns not present in nowcast

Column mapping summary:
  Original columns: 94
  Renamed columns: 79
  Dropped columns: 15
  Final columns: 79

 RangeIndex(start=0, stop=4087, step=1)
Loading evaluation data for 2025-08-24 in zone ELSPOT NO2
Dropping 15 gust columns not present in nowcast

Column mapping summary:
  Original columns: 94
  Renamed columns: 79
  Dropped columns: 15
  Final columns: 79

 RangeIndex(start=0, stop=4087, step=1)
Loading evaluation data for 2025-08-25 i

/Users/noemiweber/Documents/Studium/master/4.semester_norway/WindAIcompetition/notebooks/../src/data_processing/downtime.py:64: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  maintenance_events.drop(


Maintenance hours in operational period: 0
Preprocessing done. X_eval shape: (696, 17, 15, 10)
Dropped timestamps due to maintenance: 0
